In [4]:
import asyncio
import aiohttp
from aiohttp import ClientSession, ClientTimeout
from bs4 import BeautifulSoup
import os
import re
import lxml.html
import random

# === Configuration ===
start_idx = 0  # manually set start index
end_idx = 50  # manually set end index
base_dir = r"./nist_data"
ids_file_path = r'./nist_data/nist_ids.csv'
jdx_semaphore = asyncio.Semaphore(3)

In [5]:
# === Directories ===
folders = {
    "jdx": "jdx",
    "mol": "mol",
    "sdf": "sdf",
    "inchi": "inchi",
    "inchikey": "inchikey",
    "name": "name",
    "mw": "mw"
}

for folder in folders.values():
    os.makedirs(os.path.join(base_dir, folder), exist_ok=True)

# === URL and Regex ===
nist_url = "https://webbook.nist.gov/cgi/cbook.cgi"
id_re = re.compile(r'/cgi/cbook.cgi\?ID=(.*?)&')

# === Helper Function to Save ===
def save_file(path, content, binary=False):
    with open(path, 'wb' if binary else 'w', encoding=None if binary else 'utf-8') as f:
        f.write(content)

# === Retry-enabled fetch with HTML check ===
async def fetch_with_retries(session, url, params, binary=True, retries=5, base_delay=1):
    for attempt in range(retries):
        try:
            async with session.get(url, params=params) as resp:
                content = await (resp.read() if binary else resp.text())
                if resp.status == 429 or (binary and b'<html' in content[:100].lower()):
                    raise aiohttp.ClientResponseError(
                        resp.request_info, resp.history,
                        code=429, message="Rate limited or invalid content", headers=resp.headers
                    )
                return content
        except Exception as e:
            wait = base_delay * (2 ** attempt) + random.uniform(0.5, 1.5)
            print(f"Retry {attempt + 1}/{retries} for {params} after {wait:.1f}s due to: {e}")
            await asyncio.sleep(wait)
    raise Exception(f"Failed after {retries} retries for {params}")

# === Save with empty check ===
async def fetch_and_save(session, nistid, param, suffix, folder_key, binary=True, check_empty=False):
    path = os.path.join(base_dir, folders[folder_key], f"{nistid}{suffix}")
    if os.path.exists(path):
        print(f"{nistid}{suffix}: already exists")
        return
    try:
        content = await fetch_with_retries(session, nist_url, param, binary=binary)
        if check_empty and (not content.strip() or len(content) < 50):
            print(f"{nistid}{suffix}: empty or too short")
            return
        save_file(path, content, binary)
        print(f"{nistid}{suffix}: downloaded")
    except Exception as e:
        print(f"Error fetching {nistid}{suffix}: {e}")
        raise

# === Try JDX indices ===
async def try_jdx_indices(session, nistid, failed_list):
    async with jdx_semaphore:
        for index in range(3):
            try:
                await fetch_and_save(session, nistid, {'JCAMP': nistid, 'Type': 'IR', 'Index': index}, f'_{index}.jdx', 'jdx')
                return
            except Exception as e:
                print(f"{nistid}_index{index}.jdx failed: {e}")
        print(f"No valid JDX for {nistid}")
        failed_list.append(nistid)

# === Other Data Fetch ===
async def get_inchikey(session, nistid):
    path = os.path.join(base_dir, folders['inchikey'], f"{nistid}_inchikey.txt")
    if os.path.exists(path):
        return
    try:
        async with session.get(nist_url, params={'ID': nistid, 'Units': 'SI'}) as resp:
            raw = await resp.read()
            html = lxml.html.fromstring(raw)
            key = html.xpath('//li/span[@style="font-family: monospace;"]/text()')
            if key:
                save_file(path, f"InChIKey: {key[0]}")
    except Exception as e:
        print(f"Error fetching InChIKey for {nistid}: {e}")

async def get_name_and_formula(session, nistid):
    path = os.path.join(base_dir, folders['name'], f"{nistid}_name.txt")
    if os.path.exists(path):
        return
    try:
        async with session.get(nist_url, params={'ID': nistid, 'Units': 'SI'}) as resp:
            raw = await resp.read()
            html = lxml.html.fromstring(raw)
            name = html.xpath('//h1[@id="Top"]/text()')
            formula = html.xpath('//ul[1]/li[1]//text()')
            other = html.xpath('//li[contains(.,"Other names:")]/text()')
            with open(path, 'w', encoding='utf-8') as f:
                if name: f.write(f"Name: {name[0].strip()}")
                if formula: f.write(f"Formula: {formula[0].strip()}")
                if other: f.write(f"Other names: {other[0].strip()}")
    except Exception as e:
        print(f"Error fetching name/formula for {nistid}: {e}")

async def get_mw(session, nistid):
    path = os.path.join(base_dir, folders['mw'], f"{nistid}_mw.txt")
    if os.path.exists(path):
        return
    try:
        async with session.get(nist_url, params={'ID': nistid, 'Units': 'SI'}) as resp:
            raw = await resp.read()
            html = lxml.html.fromstring(raw)
            mw = html.xpath('//li[contains(.,"Molecular weight")]/text()')
            if mw:
                save_file(path, f"Molecular weight: {mw[0].strip()}")
    except Exception as e:
        print(f"Error fetching MW for {nistid}: {e}")

# === Fetch Compound Routine ===
async def fetch_compound_data(session, nistid, failed_jdx_list, idx):
    tasks = [
        try_jdx_indices(session, nistid, failed_jdx_list),
        fetch_and_save(session, nistid, {'Str2File': nistid}, '.mol', 'mol', check_empty=True),
        fetch_and_save(session, nistid, {'Str3File': nistid}, '.sdf', 'sdf', check_empty=True),
        fetch_and_save(session, nistid, {'GetInChI': nistid}, '.inchi', 'inchi', check_empty=True),
        get_inchikey(session, nistid),
        get_name_and_formula(session, nistid),
        get_mw(session, nistid),
    ]

    for task in tasks:
        try:
            await task
        except Exception as e:
            print(f"Error in {task.__name__ if hasattr(task, '__name__') else 'anonymous task'} for {nistid}: {e}")

    print(f"Finished index {idx}: {nistid}")


In [6]:
# === Main ===
async def main():
    timeout = ClientTimeout(total=30)
    conn = aiohttp.TCPConnector(limit=20)
    failed_jdx = []

    async with ClientSession(timeout=timeout, connector=conn) as session:
        if not os.path.exists(ids_file_path):
            print(" IDs file not found!")
            return
        with open(ids_file_path, 'r') as f:
            all_ids = [line.strip() for line in f if line.strip()]

        all_ids = all_ids[start_idx:end_idx]
        print(f"Found {len(all_ids)} IDs (from index {start_idx} to {end_idx}). Starting download...")

        tasks = [fetch_compound_data(session, nid, failed_jdx, start_idx + idx) for idx, nid in enumerate(all_ids)]
        await asyncio.gather(*tasks)

    with open(os.path.join(base_dir, "failed_jdx.txt"), "w") as f:
        for nid in failed_jdx:
            f.write(nid + "\n")

    print(f"\n All done! Processed {len(all_ids)} compounds. Failed JDX: {len(failed_jdx)}")

if __name__ == "__main__":
    await main()

Found 50 IDs (from index 0 to 50). Starting download...
B6000033_0.jdx: already exists
B6000033.mol: already exists
B6000033.sdf: already exists
B6000033.inchi: already exists
B6000034_0.jdx: already exists
B6000034.mol: already exists
B6000034.sdf: already exists
B6000034.inchi: already exists
B6000036_0.jdx: already exists
B6000036.mol: already exists
Finished index 0: B6000033
Finished index 1: B6000034
B6000036.sdf: empty or too short
B6000036.inchi: already exists
B6000056_0.jdx: downloaded
B6000058_0.jdx: downloaded
B6000057_0.jdx: downloaded
Finished index 2: B6000036
B6000056.mol: downloaded
B6000059_0.jdx: downloaded
B6000058.mol: downloaded
B6000057.mol: downloaded
B6000060_0.jdx: downloaded
B6000056.sdf: downloaded
B6000059.mol: downloaded


C:\Users\Corei5_8GBRAM_512SSD\AppData\Local\Temp\ipykernel_30400\2882928503.py:31: DeprecationWarning: code argument is deprecated, use status instead
  raise aiohttp.ClientResponseError(


Retry 1/5 for {'JCAMP': 'B6000062', 'Type': 'IR', 'Index': 0} after 1.7s due to: 429, message='Rate limited or invalid content', url='https://webbook.nist.gov/cgi/cbook.cgi?JCAMP=B6000062&Type=IR&Index=0'
B6000058.sdf: empty or too short
B6000057.sdf: downloaded
B6000060.mol: downloaded
B6000056.inchi: downloaded
B6000059.sdf: empty or too short
B6000058.inchi: downloaded
Retry 1/5 for {'JCAMP': 'B6000061', 'Type': 'IR', 'Index': 0} after 1.7s due to: 429, message='Rate limited or invalid content', url='https://webbook.nist.gov/cgi/cbook.cgi?JCAMP=B6000061&Type=IR&Index=0'
B6000057.inchi: downloaded
B6000060.sdf: empty or too short
B6000059.inchi: downloaded
B6000060.inchi: downloaded
Retry 1/5 for {'JCAMP': 'B6000063', 'Type': 'IR', 'Index': 0} after 1.9s due to: 429, message='Rate limited or invalid content', url='https://webbook.nist.gov/cgi/cbook.cgi?JCAMP=B6000063&Type=IR&Index=0'
Finished index 3: B6000056
Finished index 5: B6000058
Finished index 6: B6000059
Finished index 4: B6